ConvNextで過学習対策

重み減衰（Adam→AdamW）,Dropout←これがないとAdamWの効果が薄れるらしい

In [5]:
from sklearn.preprocessing import StandardScaler
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import math

# --- 設定: ファイルパス ---
pre = "/home/hiromu/energy"
IMG_ROOT = f'{pre}/data/1201_humomentstest'
TRAIN_CSV_PATH = f'{pre}/src/FF/AFF/dataset_upsamp/train_max_upsampled.csv'
VAL_CSV_PATH   = f'{pre}/src/FF/AFF/dataset_upsamp/val_fixed.csv'

# 学習設定
TARGET_EPOCH = 100
SAVE_DIR = f"../save/OFtest/ConvNext_{TARGET_EPOCH}epoch_fixed_dataset"

# --- 1. CBAM モジュール (初期値 5:5 版) ---
class CBAM(nn.Module):
    def __init__(self, channels, reduction=16, initial_weight=0.5):
        super(CBAM, self).__init__()
        reduced_channels = max(1, channels // reduction)
        self.fc = nn.Sequential(
            nn.Linear(channels, reduced_channels),
            nn.ReLU(inplace=True),
            nn.Linear(reduced_channels, channels)
        )
        self.sigmoid_channel = nn.Sigmoid()
        self.conv_spatial = nn.Conv2d(2, 1, kernel_size=7, padding=3)
        self.sigmoid_spatial = nn.Sigmoid()
        
        # 初期値を指定 (0.5 の場合、内部のロジットは 0.0 になります)
        self._initialize_weights(initial_weight)

    def _initialize_weights(self, target_prob):
        logit_value = math.log(target_prob / (1 - target_prob))
        with torch.no_grad():
            nn.init.constant_(self.conv_spatial.bias, logit_value)
            self.conv_spatial.weight.data *= 0.01 

    def forward(self, x):
        avg_out = self.fc(torch.mean(x, dim=(2, 3)))
        max_out = self.fc(torch.amax(x, dim=(2, 3)))
        ch_att = self.sigmoid_channel(avg_out + max_out).unsqueeze(-1).unsqueeze(-1)
        
        avg_out_sp = torch.mean(x, dim=1, keepdim=True)
        max_out_sp = torch.amax(x, dim=1, keepdim=True)
        sp_att = self.sigmoid_spatial(self.conv_spatial(torch.cat([avg_out_sp, max_out_sp], dim=1)))
        
        return ch_att * sp_att

# --- 2. 融合モデル ---
class FusionModel(nn.Module):
    def __init__(self, num_classes):
        super(FusionModel, self).__init__()
        convnext = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.DEFAULT)
        self.dl_extractor = convnext.features
        dl_out_channels = 768
        
        self.hc_projector = nn.Sequential(
            nn.Linear(10, dl_out_channels),
            nn.BatchNorm1d(dl_out_channels),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.3)
        )
        
        self.dl_norm = nn.BatchNorm2d(dl_out_channels)
        # ここも明示的に 0.5 (5:5) を指定
        self.fusion_attention = CBAM(dl_out_channels, initial_weight=0.5) 
        
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.BatchNorm1d(dl_out_channels),
            nn.Dropout(p=0.4),
            nn.Linear(dl_out_channels, num_classes)
        )

    def forward(self, img, hc_vec):
        Y = self.dl_extractor(img)
        Y = self.dl_norm(Y)
        B, C, H, W = Y.shape
        X = self.hc_projector(hc_vec).view(B, C, 1, 1).expand(B, C, H, W)
        
        weights = self.fusion_attention(X + Y)
        fused = weights * X + (1 - weights) * Y
        
        return self.classifier(fused)

# --- 3. Datasetクラス ---
class CustomMultiModalDataset(Dataset):
    def __init__(self, dataframe, img_root, scaler=None, transform=None, is_train=True):
        self.df = dataframe.reset_index(drop=True)
        self.img_root = img_root
        self.transform = transform
        self.feature_cols = ['h0', 'h1', 'h2', 'h3', 'h4', 'h5', 'h6', 'h7', 'size_count', 'R']
        features = self.df[self.feature_cols].values
        if is_train:
            self.scaler = StandardScaler()
            self.hc_features = self.scaler.fit_transform(features).astype('float32')
        else:
            assert scaler is not None
            self.scaler = scaler
            self.hc_features = self.scaler.transform(features).astype('float32')
        self.labels = self.df['Label'].values

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_root, 'mask', row['category'], row['filename'])
        try: image = Image.open(img_path).convert('RGB')
        except: image = Image.new('RGB', (224, 224), (0, 0, 0))
        if self.transform: image = self.transform(image)
        hc_feat = torch.tensor(self.hc_features[idx])
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return image, hc_feat, label

# --- 4. ユーティリティ (アーリーストッピング削除) ---
def save_learning_curve(history, save_path):
    epochs = range(1, len(history['train_loss']) + 1)
    plt.figure(figsize=(14, 5))

    # Lossの推移 (Train vs Val)
    plt.subplot(1, 2, 1)
    plt.plot(epochs, history['train_loss'], 'r-', label='Train Loss')
    plt.plot(epochs, history['val_loss'], 'g-', label='Val Loss')
    plt.title('Training & Validation Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)

    # Accuracyの推移 (Val)
    plt.subplot(1, 2, 2)
    plt.plot(epochs, history['val_acc'], 'b-', label='Val Acc')
    plt.title('Validation Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()

# --- 5. メイン学習関数 ---
def train_model():
    BATCH_SIZE = 8
    EPOCHS = TARGET_EPOCH
    LR = 0.0001
    WD = 0.05
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    os.makedirs(SAVE_DIR, exist_ok=True)
    best_model_path = f'{SAVE_DIR}/best_fusion_model.pth'

    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    train_df, val_df = pd.read_csv(TRAIN_CSV_PATH), pd.read_csv(VAL_CSV_PATH)
    train_ds = CustomMultiModalDataset(train_df, IMG_ROOT, transform=transform, is_train=True)
    val_ds = CustomMultiModalDataset(val_df, IMG_ROOT, scaler=train_ds.scaler, transform=transform, is_train=False)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
    
    model = FusionModel(num_classes=3).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    
    # 学習率スケジューラ
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    
    history = {'train_loss': [], 'val_loss': [], 'val_acc': []}
    best_acc = 0.0 # ベスト精度を記録する変数

    print(f"Starting Training on {DEVICE} without Early Stopping...")

    for epoch in range(EPOCHS):
        # --- Train ---
        model.train()
        running_loss = 0.0
        for images, hc_feats, labels in train_loader:
            images, hc_feats, labels = images.to(DEVICE), hc_feats.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(images, hc_feats), labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * images.size(0)
        
        # スケジュール更新
        scheduler.step()
        epoch_train_loss = running_loss / len(train_ds)

        # --- Val ---
        model.eval()
        v_running_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for images, hc_feats, labels in val_loader:
                images, hc_feats, labels = images.to(DEVICE), hc_feats.to(DEVICE), labels.to(DEVICE)
                outputs = model(images, hc_feats)
                v_loss = criterion(outputs, labels)
                v_running_loss += v_loss.item() * images.size(0)
                _, preds = torch.max(outputs, 1)
                total += labels.size(0)
                correct += torch.sum(preds == labels.data)
        
        epoch_val_loss = v_running_loss / len(val_ds)
        val_acc = (correct.double() / total).item()
        
        history['train_loss'].append(epoch_train_loss)
        history['val_loss'].append(epoch_val_loss)
        history['val_acc'].append(val_acc)
        
        curr_lr = optimizer.param_groups[0]['lr']
        print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f} | Val Acc: {val_acc:.4f} | LR: {curr_lr:.6f}")
        
        # シンプルに最高精度が出たときだけ保存
        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), best_model_path)
            print(f"  --> Best Model Saved! (Acc: {best_acc:.4f})")
            
        if (epoch + 1) % 5 == 0: 
            save_learning_curve(history, f'{SAVE_DIR}/learning_curve.png')

    print(f"Training Complete. Best Val Acc: {best_acc:.4f}")
    save_learning_curve(history, f'{SAVE_DIR}/learning_curve.png')
    return history

if __name__ == "__main__":
    train_model()

Starting Training on cuda without Early Stopping...
Epoch 1/100 | Train Loss: 1.1417 | Val Loss: 1.0643 | Val Acc: 0.4444 | LR: 0.000100
  --> Best Model Saved! (Acc: 0.4444)
Epoch 2/100 | Train Loss: 0.9935 | Val Loss: 0.9680 | Val Acc: 0.5208 | LR: 0.000100
  --> Best Model Saved! (Acc: 0.5208)
Epoch 3/100 | Train Loss: 0.9058 | Val Loss: 0.9804 | Val Acc: 0.5208 | LR: 0.000100
Epoch 4/100 | Train Loss: 0.7667 | Val Loss: 1.0656 | Val Acc: 0.5000 | LR: 0.000100
Epoch 5/100 | Train Loss: 0.6448 | Val Loss: 1.2881 | Val Acc: 0.4583 | LR: 0.000099
Epoch 6/100 | Train Loss: 0.5527 | Val Loss: 1.1937 | Val Acc: 0.4792 | LR: 0.000099
Epoch 7/100 | Train Loss: 0.4361 | Val Loss: 1.3321 | Val Acc: 0.4861 | LR: 0.000099
Epoch 8/100 | Train Loss: 0.4123 | Val Loss: 1.3569 | Val Acc: 0.4653 | LR: 0.000098
Epoch 9/100 | Train Loss: 0.2621 | Val Loss: 1.4814 | Val Acc: 0.5417 | LR: 0.000098
  --> Best Model Saved! (Acc: 0.5417)
Epoch 10/100 | Train Loss: 0.2104 | Val Loss: 1.7012 | Val Acc: 0.479

上記＋学習率スケジューラ

In [8]:
from sklearn.preprocessing import StandardScaler
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import math

# --- 設定: ファイルパス ---
pre = "/home/hiromu/energy"
IMG_ROOT = f'{pre}/data/1201_humomentstest'
TRAIN_CSV_PATH = f'{pre}/src/FF/AFF/dataset_upsamp/train_max_upsampled.csv'
VAL_CSV_PATH   = f'{pre}/src/FF/AFF/dataset_upsamp/val_fixed.csv'

# 学習設定
TARGET_EPOCH = 100
SAVE_DIR = f"../save/OFtest/ConvNext_{TARGET_EPOCH}epoch_fixed_dataset"

# --- 1. CBAM モジュール ---
class CBAM(nn.Module):
    def __init__(self, channels, reduction=16, initial_weight=0.5):
        super(CBAM, self).__init__()
        reduced_channels = max(1, channels // reduction)
        self.fc = nn.Sequential(
            nn.Linear(channels, reduced_channels),
            nn.ReLU(inplace=True),
            nn.Linear(reduced_channels, channels)
        )
        self.sigmoid_channel = nn.Sigmoid()
        self.conv_spatial = nn.Conv2d(2, 1, kernel_size=7, padding=3)
        self.sigmoid_spatial = nn.Sigmoid()
        self._initialize_weights(initial_weight)

    def _initialize_weights(self, target_prob):
        logit_value = math.log(target_prob / (1 - target_prob))
        with torch.no_grad():
            nn.init.constant_(self.conv_spatial.bias, logit_value)
            self.conv_spatial.weight.data *= 0.01 

    def forward(self, x):
        avg_out = self.fc(torch.mean(x, dim=(2, 3)))
        max_out = self.fc(torch.amax(x, dim=(2, 3)))
        ch_att = self.sigmoid_channel(avg_out + max_out).unsqueeze(-1).unsqueeze(-1)
        avg_out_sp = torch.mean(x, dim=1, keepdim=True)
        max_out_sp = torch.amax(x, dim=1, keepdim=True)
        sp_att = self.sigmoid_spatial(self.conv_spatial(torch.cat([avg_out_sp, max_out_sp], dim=1)))
        return ch_att * sp_att

# --- 2. 融合モデル ---
class FusionModel(nn.Module):
    def __init__(self, num_classes):
        super(FusionModel, self).__init__()
        convnext = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.DEFAULT)
        self.dl_extractor = convnext.features
        dl_out_channels = 768
        self.hc_projector = nn.Sequential(
            nn.Linear(10, dl_out_channels),
            nn.BatchNorm1d(dl_out_channels),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.3)
        )
        self.dl_norm = nn.BatchNorm2d(dl_out_channels)
        self.fusion_attention = CBAM(dl_out_channels, initial_weight=0.5)
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.BatchNorm1d(dl_out_channels),
            nn.Dropout(p=0.4),
            nn.Linear(dl_out_channels, num_classes)
        )

    def forward(self, img, hc_vec):
        Y = self.dl_extractor(img)
        Y = self.dl_norm(Y)
        B, C, H, W = Y.shape
        X = self.hc_projector(hc_vec).view(B, C, 1, 1).expand(B, C, H, W)
        weights = self.fusion_attention(X + Y)
        fused = weights * X + (1 - weights) * Y
        return self.classifier(fused)

# --- 3. Datasetクラス ---
class CustomMultiModalDataset(Dataset):
    def __init__(self, dataframe, img_root, scaler=None, transform=None, is_train=True):
        self.df = dataframe.reset_index(drop=True)
        self.img_root = img_root
        self.transform = transform
        self.feature_cols = ['h0', 'h1', 'h2', 'h3', 'h4', 'h5', 'h6', 'h7', 'size_count', 'R']
        features = self.df[self.feature_cols].values
        if is_train:
            self.scaler = StandardScaler()
            self.hc_features = self.scaler.fit_transform(features).astype('float32')
        else:
            assert scaler is not None
            self.scaler = scaler
            self.hc_features = self.scaler.transform(features).astype('float32')
        self.labels = self.df['Label'].values

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_root, 'mask', row['category'], row['filename'])
        try: image = Image.open(img_path).convert('RGB')
        except: image = Image.new('RGB', (224, 224), (0, 0, 0))
        if self.transform: image = self.transform(image)
        hc_feat = torch.tensor(self.hc_features[idx])
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return image, hc_feat, label

# --- 4. ユーティリティ ---
def save_learning_curve(history, save_path):
    epochs = range(1, len(history['train_loss']) + 1)
    plt.figure(figsize=(14, 5))

    # Lossの推移 (Train vs Val)
    plt.subplot(1, 2, 1)
    plt.plot(epochs, history['train_loss'], 'r-', label='Train Loss')
    plt.plot(epochs, history['val_loss'], 'g-', label='Val Loss') # 追加
    plt.title('Training & Validation Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)

    # Accuracyの推移 (Val)
    plt.subplot(1, 2, 2)
    plt.plot(epochs, history['val_acc'], 'b-', label='Val Acc')
    plt.title('Validation Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()

class EarlyStopping:
    def __init__(self, patience=7, verbose=False, delta=0, path='checkpoint.pth'):
        self.patience, self.verbose, self.delta, self.path = patience, verbose, delta, path
        self.counter, self.best_score, self.early_stop, self.val_acc_max = 0, None, False, 0

    def __call__(self, val_acc, model):
        score = val_acc
        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_acc, model)
        elif score < self.best_score + self.delta:
            self.counter += 1
            if self.verbose: print(f'EarlyStopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience: self.early_stop = True
        else:
            self.best_score = score
            self.save_checkpoint(val_acc, model)
            self.counter = 0

    def save_checkpoint(self, val_acc, model):
        if self.verbose: print(f'Validation accuracy increased ({self.val_acc_max:.4f} --> {val_acc:.4f}). Saving...')
        torch.save(model.state_dict(), self.path)
        self.val_acc_max = val_acc

# --- 5. メイン学習関数 ---
def train_model():
    BATCH_SIZE, EPOCHS, LR, WD, PATIENCE = 8, TARGET_EPOCH, 0.0001, 0.05, 100
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    os.makedirs(SAVE_DIR, exist_ok=True)
    best_model_path = f'{SAVE_DIR}/best_fusion_model.pth'

    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    train_df, val_df = pd.read_csv(TRAIN_CSV_PATH), pd.read_csv(VAL_CSV_PATH)
    train_ds = CustomMultiModalDataset(train_df, IMG_ROOT, transform=transform, is_train=True)
    val_ds = CustomMultiModalDataset(val_df, IMG_ROOT, scaler=train_ds.scaler, transform=transform, is_train=False)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
    
    model = FusionModel(num_classes=3).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    
    # ★ 学習率スケジューラ: CosineAnnealing
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    
    early_stopping = EarlyStopping(patience=PATIENCE, verbose=True, path=best_model_path)
    history = {'train_loss': [], 'val_loss': [], 'val_acc': []}

    print(f"Starting Training on {DEVICE}...")

    for epoch in range(EPOCHS):
        # --- Train ---
        model.train()
        running_loss = 0.0
        for images, hc_feats, labels in train_loader:
            images, hc_feats, labels = images.to(DEVICE), hc_feats.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(images, hc_feats), labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * images.size(0)
        
        # ★ スケジュール更新
        scheduler.step()
        epoch_train_loss = running_loss / len(train_ds)

        # --- Val ---
        model.eval()
        v_running_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for images, hc_feats, labels in val_loader:
                images, hc_feats, labels = images.to(DEVICE), hc_feats.to(DEVICE), labels.to(DEVICE)
                outputs = model(images, hc_feats)
                v_loss = criterion(outputs, labels) # ★ 検証損失の計算
                v_running_loss += v_loss.item() * images.size(0)
                _, preds = torch.max(outputs, 1)
                total += labels.size(0)
                correct += torch.sum(preds == labels.data)
        
        epoch_val_loss = v_running_loss / len(val_ds)
        val_acc = (correct.double() / total).item()
        
        history['train_loss'].append(epoch_train_loss)
        history['val_loss'].append(epoch_val_loss)
        history['val_acc'].append(val_acc)
        
        curr_lr = optimizer.param_groups[0]['lr']
        print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f} | Val Acc: {val_acc:.4f} | LR: {curr_lr:.6f}")
        
        early_stopping(val_acc, model)
        if early_stopping.early_stop: break
        if (epoch + 1) % 5 == 0: save_learning_curve(history, f'{SAVE_DIR}/learning_curve.png')

    save_learning_curve(history, f'{SAVE_DIR}/learning_curve.png')
    return history

if __name__ == "__main__":
    train_model()

Starting Training on cuda...
Epoch 1/100 | Train Loss: 1.1206 | Val Loss: 0.9165 | Val Acc: 0.5972 | LR: 0.000100
Validation accuracy increased (0.0000 --> 0.5972). Saving...
Epoch 2/100 | Train Loss: 0.9748 | Val Loss: 1.0864 | Val Acc: 0.4722 | LR: 0.000100
EarlyStopping counter: 1 out of 100
Epoch 3/100 | Train Loss: 0.8683 | Val Loss: 1.0451 | Val Acc: 0.5278 | LR: 0.000100
EarlyStopping counter: 2 out of 100
Epoch 4/100 | Train Loss: 0.7427 | Val Loss: 1.0406 | Val Acc: 0.5903 | LR: 0.000100
EarlyStopping counter: 3 out of 100
Epoch 5/100 | Train Loss: 0.6579 | Val Loss: 1.2773 | Val Acc: 0.4722 | LR: 0.000099
EarlyStopping counter: 4 out of 100
Epoch 6/100 | Train Loss: 0.5233 | Val Loss: 1.3133 | Val Acc: 0.4722 | LR: 0.000099
EarlyStopping counter: 5 out of 100
Epoch 7/100 | Train Loss: 0.4146 | Val Loss: 1.4221 | Val Acc: 0.5347 | LR: 0.000099
EarlyStopping counter: 6 out of 100
Epoch 8/100 | Train Loss: 0.3659 | Val Loss: 1.2130 | Val Acc: 0.5347 | LR: 0.000098
EarlyStopping 

上記＋データ拡張

In [ ]:
from sklearn.preprocessing import StandardScaler
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import math

# --- 設定: ファイルパス ---
pre = "/home/hiromu/energy"
IMG_ROOT = f'{pre}/data/1201_humomentstest'
TRAIN_CSV_PATH = f'{pre}/src/FF/AFF/dataset_upsamp/train_max_upsampled.csv'
VAL_CSV_PATH   = f'{pre}/src/FF/AFF/dataset_upsamp/val_fixed.csv'

# 学習設定
TARGET_EPOCH = 100
SAVE_DIR = f"../save/OFtest/ConvNext_{TARGET_EPOCH}epoch_fixed_dataset"

# ★ スケジューラのオン/オフを切り替えるフラグ
USE_SCHEDULER = False # Trueでコサイン波状に学習率を落とす、Falseで固定

# --- 1. CBAM モジュール (初期値 5:5 版) ---
class CBAM(nn.Module):
    def __init__(self, channels, reduction=16, initial_weight=0.5):
        super(CBAM, self).__init__()
        reduced_channels = max(1, channels // reduction)
        self.fc = nn.Sequential(
            nn.Linear(channels, reduced_channels),
            nn.ReLU(inplace=True),
            nn.Linear(reduced_channels, channels)
        )
        self.sigmoid_channel = nn.Sigmoid()
        self.conv_spatial = nn.Conv2d(2, 1, kernel_size=7, padding=3)
        self.sigmoid_spatial = nn.Sigmoid()
        self._initialize_weights(initial_weight)

    def _initialize_weights(self, target_prob):
        logit_value = math.log(target_prob / (1 - target_prob))
        with torch.no_grad():
            nn.init.constant_(self.conv_spatial.bias, logit_value)
            self.conv_spatial.weight.data *= 0.01 

    def forward(self, x):
        avg_out = self.fc(torch.mean(x, dim=(2, 3)))
        max_out = self.fc(torch.amax(x, dim=(2, 3)))
        ch_att = self.sigmoid_channel(avg_out + max_out).unsqueeze(-1).unsqueeze(-1)
        
        avg_out_sp = torch.mean(x, dim=1, keepdim=True)
        max_out_sp = torch.amax(x, dim=1, keepdim=True)
        sp_att = self.sigmoid_spatial(self.conv_spatial(torch.cat([avg_out_sp, max_out_sp], dim=1)))
        
        return ch_att * sp_att

# --- 2. 融合モデル ---
class FusionModel(nn.Module):
    def __init__(self, num_classes):
        super(FusionModel, self).__init__()
        convnext = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.DEFAULT)
        self.dl_extractor = convnext.features
        dl_out_channels = 768
        
        self.hc_projector = nn.Sequential(
            nn.Linear(10, dl_out_channels),
            nn.BatchNorm1d(dl_out_channels),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.3)
        )
        
        self.dl_norm = nn.BatchNorm2d(dl_out_channels)
        self.fusion_attention = CBAM(dl_out_channels, initial_weight=0.5) 
        
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.BatchNorm1d(dl_out_channels),
            nn.Dropout(p=0.4),
            nn.Linear(dl_out_channels, num_classes)
        )

    def forward(self, img, hc_vec):
        Y = self.dl_extractor(img)
        Y = self.dl_norm(Y)
        B, C, H, W = Y.shape
        X = self.hc_projector(hc_vec).view(B, C, 1, 1).expand(B, C, H, W)
        
        weights = self.fusion_attention(X + Y)
        fused = weights * X + (1 - weights) * Y
        
        return self.classifier(fused)

# --- 3. Datasetクラス ---
class CustomMultiModalDataset(Dataset):
    def __init__(self, dataframe, img_root, scaler=None, transform=None, is_train=True):
        self.df = dataframe.reset_index(drop=True)
        self.img_root = img_root
        self.transform = transform
        self.feature_cols = ['h0', 'h1', 'h2', 'h3', 'h4', 'h5', 'h6', 'h7', 'size_count', 'R']
        features = self.df[self.feature_cols].values
        if is_train:
            self.scaler = StandardScaler()
            self.hc_features = self.scaler.fit_transform(features).astype('float32')
        else:
            assert scaler is not None
            self.scaler = scaler
            self.hc_features = self.scaler.transform(features).astype('float32')
        self.labels = self.df['Label'].values

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_root, 'mask', row['category'], row['filename'])
        try: image = Image.open(img_path).convert('RGB')
        except: image = Image.new('RGB', (224, 224), (0, 0, 0))
        if self.transform: image = self.transform(image)
        hc_feat = torch.tensor(self.hc_features[idx])
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return image, hc_feat, label

# --- 4. ユーティリティ ---
def save_learning_curve(history, save_path):
    epochs = range(1, len(history['train_loss']) + 1)
    plt.figure(figsize=(14, 5))

    # Lossの推移 (Train vs Val)
    plt.subplot(1, 2, 1)
    plt.plot(epochs, history['train_loss'], 'r-', label='Train Loss')
    plt.plot(epochs, history['val_loss'], 'g-', label='Val Loss')
    plt.title('Training & Validation Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)

    # Accuracyの推移 (Val)
    plt.subplot(1, 2, 2)
    plt.plot(epochs, history['val_acc'], 'b-', label='Val Acc')
    plt.title('Validation Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()

# --- 5. メイン学習関数 ---
def train_model():
    BATCH_SIZE = 8
    EPOCHS = TARGET_EPOCH
    LR = 0.0001
    WD = 0.05
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    os.makedirs(SAVE_DIR, exist_ok=True)
    best_model_path = f'{SAVE_DIR}/best_fusion_model.pth'

    # 訓練用 (データ拡張あり)
    train_transform = transforms.Compose([
        transforms.Resize((236, 236)),
        transforms.RandomRotation(degrees=15),
        transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    # 検証・テスト用 (データ拡張なし)
    val_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    train_df, val_df = pd.read_csv(TRAIN_CSV_PATH), pd.read_csv(VAL_CSV_PATH)
    
    train_ds = CustomMultiModalDataset(train_df, IMG_ROOT, transform=train_transform, is_train=True)
    val_ds = CustomMultiModalDataset(val_df, IMG_ROOT, scaler=train_ds.scaler, transform=val_transform, is_train=False)
    
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
    
    model = FusionModel(num_classes=3).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    
    # ★ フラグに基づいてスケジューラを初期化
    if USE_SCHEDULER:
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
        print("Scheduler: ENABLED (Cosine Annealing)")
    else:
        scheduler = None
        print("Scheduler: DISABLED (Fixed Learning Rate)")
    
    history = {'train_loss': [], 'val_loss': [], 'val_acc': []}
    best_acc = 0.0

    print(f"Starting Training on {DEVICE}...")

    for epoch in range(EPOCHS):
        # --- Train ---
        model.train()
        running_loss = 0.0
        for images, hc_feats, labels in train_loader:
            images, hc_feats, labels = images.to(DEVICE), hc_feats.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(images, hc_feats), labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * images.size(0)
        
        # ★ スケジューラが有効な場合のみステップを進める
        if scheduler is not None:
            scheduler.step()
            
        epoch_train_loss = running_loss / len(train_ds)

        # --- Val ---
        model.eval()
        v_running_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for images, hc_feats, labels in val_loader:
                images, hc_feats, labels = images.to(DEVICE), hc_feats.to(DEVICE), labels.to(DEVICE)
                outputs = model(images, hc_feats)
                v_loss = criterion(outputs, labels)
                v_running_loss += v_loss.item() * images.size(0)
                _, preds = torch.max(outputs, 1)
                total += labels.size(0)
                correct += torch.sum(preds == labels.data)
        
        epoch_val_loss = v_running_loss / len(val_ds)
        val_acc = (correct.double() / total).item()
        
        history['train_loss'].append(epoch_train_loss)
        history['val_loss'].append(epoch_val_loss)
        history['val_acc'].append(val_acc)
        
        curr_lr = optimizer.param_groups[0]['lr']
        print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f} | Val Acc: {val_acc:.4f} | LR: {curr_lr:.6f}")
        
        # ベストモデルの保存
        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), best_model_path)
            print(f"  --> Best Model Saved! (Acc: {best_acc:.4f})")
            
        if (epoch + 1) % 5 == 0: 
            save_learning_curve(history, f'{SAVE_DIR}/learning_curve.png')

    print(f"Training Complete. Best Val Acc: {best_acc:.4f}")
    save_learning_curve(history, f'{SAVE_DIR}/learning_curve.png')
    return history

if __name__ == "__main__":
    train_model()

Scheduler: DISABLED (Fixed Learning Rate)
Starting Training on cuda...
Epoch 1/100 | Train Loss: 1.1690 | Val Loss: 0.9846 | Val Acc: 0.4583 | LR: 0.000100
  --> Best Model Saved! (Acc: 0.4583)
Epoch 2/100 | Train Loss: 1.0527 | Val Loss: 0.9885 | Val Acc: 0.4583 | LR: 0.000100
Epoch 3/100 | Train Loss: 1.0001 | Val Loss: 0.8939 | Val Acc: 0.6042 | LR: 0.000100
  --> Best Model Saved! (Acc: 0.6042)
Epoch 4/100 | Train Loss: 0.9759 | Val Loss: 0.8403 | Val Acc: 0.6042 | LR: 0.000100
Epoch 5/100 | Train Loss: 0.9576 | Val Loss: 0.8251 | Val Acc: 0.6597 | LR: 0.000100
  --> Best Model Saved! (Acc: 0.6597)
Epoch 6/100 | Train Loss: 0.9638 | Val Loss: 0.8461 | Val Acc: 0.5625 | LR: 0.000100
Epoch 7/100 | Train Loss: 0.9693 | Val Loss: 0.8334 | Val Acc: 0.6389 | LR: 0.000100
Epoch 8/100 | Train Loss: 0.9229 | Val Loss: 0.8389 | Val Acc: 0.5625 | LR: 0.000100
Epoch 9/100 | Train Loss: 0.9179 | Val Loss: 0.8318 | Val Acc: 0.6181 | LR: 0.000100
Epoch 10/100 | Train Loss: 0.9416 | Val Loss: 0.81

In [13]:
from sklearn.preprocessing import StandardScaler
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from torchvision.ops.stochastic_depth import StochasticDepth # DropPath用
from PIL import Image
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import math
from collections import OrderedDict

# --- 設定: ファイルパス ---
pre = "/home/hiromu/energy"
IMG_ROOT = f'{pre}/data/26_0413_clean'
TRAIN_CSV_PATH = f'{pre}/src/FF/AFF/dataset_upsamp/train_max_upsampled.csv'
VAL_CSV_PATH   = f'{pre}/src/FF/AFF/dataset_upsamp/val_fixed.csv'

# 学習設定
TARGET_EPOCH = 100
SAVE_DIR = f"../save/OFtest/ConvNext_{TARGET_EPOCH}epoch_custom_cnn"

# ★ フラグ設定
USE_SCHEDULER = False # Trueでコサイン波状に学習率を落とす、Falseで固定
CNN_INNER_DROPOUT = 0.2 # ★ NEW: ConvNeXt内部のブロックごとに適用するDropout確率

# --- 1. カスタム ConvNeXt 実装 (完全互換版) ---

# ConvNeXt特有のチャンネル次元LayerNorm
class LayerNorm2d(nn.LayerNorm):
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.permute(0, 2, 3, 1)
        x = nn.functional.layer_norm(x, self.normalized_shape, self.weight, self.bias, self.eps)
        x = x.permute(0, 3, 1, 2)
        return x

# 次元入れ替え用ヘルパー
class Permute(nn.Module):
    def __init__(self, dims):
        super().__init__()
        self.dims = dims
    def forward(self, x):
        return torch.permute(x, self.dims)

# ConvNeXtの基本ブロック (完全互換版・修正済)
class CNBlock(nn.Module):
    def __init__(self, dim, layer_scale: float, stochastic_depth_prob: float, dropout_p: float):
        super().__init__()
        # OrderedDict を使って、公式と同じレイヤー番号(名前)を強制的に付ける
        self.block = nn.Sequential(OrderedDict([
            ("0", nn.Conv2d(dim, dim, kernel_size=7, padding=3, groups=dim, bias=True)), # Depthwise
            ("1", Permute([0, 2, 3, 1])),                # チャンネルを一番最後に移動
            ("2", nn.LayerNorm(dim, eps=1e-6)),          # LayerNorm
            ("3", nn.Linear(dim, 4 * dim, bias=True)),   # 膨張
            ("4", nn.GELU()),                            # 活性化
            ("custom_drop", nn.Dropout(p=dropout_p)),    # ★ カスタムDropout
            ("5", nn.Linear(4 * dim, dim, bias=True)),   # 収縮
        ]))
        
        # ★ 修正1: 公式の重みデータと同じ [dim, 1, 1] の形に戻す
        self.layer_scale = nn.Parameter(torch.ones(dim, 1, 1) * layer_scale)
        self.stoch_depth = StochasticDepth(stochastic_depth_prob, "row")

    def forward(self, input):
        result = self.block(input)
        
        # ★ 修正2: layer_scaleを掛ける「前」に元の画像形式 [B, C, H, W] に戻す
        result = result.permute(0, 3, 1, 2)
        
        result = self.layer_scale * result
        result = self.stoch_depth(result)
        result += input # 残差接続
        return result

# 全体の特徴抽出器を構築する関数
def create_custom_convnext_features(pretrained=True, dropout_p=0.3):
    # ConvNeXt-Tinyの公式仕様
    block_setting = [3, 3, 9, 3]
    dims = [96, 192, 384, 768]
    stoch_depth_prob = 0.1 
    
    features = nn.Sequential()
    
    # 0. Stem層
    features.add_module("0", nn.Sequential(
        nn.Conv2d(3, dims[0], kernel_size=4, stride=4),
        LayerNorm2d(dims[0], eps=1e-6)
    ))
    
    curr_stage = 0
    dp_rates = [x.item() for x in torch.linspace(0, stoch_depth_prob, sum(block_setting))]
    
    # 1. 各Stageの構築
    for i in range(4):
        # Stage間のダウンサンプリング
        if i > 0:
            features.add_module(str(2 * i), nn.Sequential(
                LayerNorm2d(dims[i-1], eps=1e-6),
                nn.Conv2d(dims[i-1], dims[i], kernel_size=2, stride=2)
            ))
        
        # ConvNeXtブロックの積み重ね
        stage_blocks = []
        for j in range(block_setting[i]):
            stage_blocks.append(CNBlock(dims[i], 1e-6, dp_rates[curr_stage], dropout_p))
            curr_stage += 1
        features.add_module(str(2 * i + 1), nn.Sequential(*stage_blocks))

    # 事前学習済み重みのロード
    if pretrained:
        print("Loading official ConvNeXt-Tiny pre-trained weights...")
        orig_model = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.DEFAULT)
        # 構造を完全に合わせたので、エラーなくスッと重みが入り、custom_dropだけが無視されます
        features.load_state_dict(orig_model.features.state_dict(), strict=False)
        
    return features


# --- 2. CBAM モジュール ---
class CBAM(nn.Module):
    def __init__(self, channels, reduction=16, initial_weight=0.5):
        super(CBAM, self).__init__()
        reduced_channels = max(1, channels // reduction)
        self.fc = nn.Sequential(
            nn.Linear(channels, reduced_channels),
            nn.ReLU(inplace=True),
            nn.Linear(reduced_channels, channels)
        )
        self.sigmoid_channel = nn.Sigmoid()
        self.conv_spatial = nn.Conv2d(2, 1, kernel_size=7, padding=3)
        self.sigmoid_spatial = nn.Sigmoid()
        self._initialize_weights(initial_weight)

    def _initialize_weights(self, target_prob):
        logit_value = math.log(target_prob / (1 - target_prob))
        with torch.no_grad():
            nn.init.constant_(self.conv_spatial.bias, logit_value)
            self.conv_spatial.weight.data *= 0.01 

    def forward(self, x):
        avg_out = self.fc(torch.mean(x, dim=(2, 3)))
        max_out = self.fc(torch.amax(x, dim=(2, 3)))
        ch_att = self.sigmoid_channel(avg_out + max_out).unsqueeze(-1).unsqueeze(-1)
        
        avg_out_sp = torch.mean(x, dim=1, keepdim=True)
        max_out_sp = torch.amax(x, dim=1, keepdim=True)
        sp_att = self.sigmoid_spatial(self.conv_spatial(torch.cat([avg_out_sp, max_out_sp], dim=1)))
        
        return ch_att * sp_att

# --- 3. 融合モデル ---
class FusionModel(nn.Module):
    def __init__(self, num_classes):
        super(FusionModel, self).__init__()
        
        # ★ ここをカスタムConvNeXtに差し替え
        self.dl_extractor = create_custom_convnext_features(pretrained=True, dropout_p=CNN_INNER_DROPOUT)
        dl_out_channels = 768
        
        self.hc_projector = nn.Sequential(
            nn.Linear(10, dl_out_channels),
            nn.BatchNorm1d(dl_out_channels),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.2)
        )
        
        self.dl_norm = nn.BatchNorm2d(dl_out_channels)
        self.fusion_attention = CBAM(dl_out_channels, initial_weight=0.5) 
        
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.BatchNorm1d(dl_out_channels),
            nn.Dropout(p=0.3),
            nn.Linear(dl_out_channels, num_classes)
        )

    def forward(self, img, hc_vec):
        Y = self.dl_extractor(img)
        Y = self.dl_norm(Y)
        B, C, H, W = Y.shape
        X = self.hc_projector(hc_vec).view(B, C, 1, 1).expand(B, C, H, W)
        
        weights = self.fusion_attention(X + Y)
        fused = weights * X + (1 - weights) * Y
        
        return self.classifier(fused)

# --- 4. Datasetクラス ---
class CustomMultiModalDataset(Dataset):
    def __init__(self, dataframe, img_root, scaler=None, transform=None, is_train=True):
        self.df = dataframe.reset_index(drop=True)
        self.img_root = img_root
        self.transform = transform
        self.feature_cols = ['h0', 'h1', 'h2', 'h3', 'h4', 'h5', 'h6', 'h7', 'size_count', 'R']
        features = self.df[self.feature_cols].values
        if is_train:
            self.scaler = StandardScaler()
            self.hc_features = self.scaler.fit_transform(features).astype('float32')
        else:
            assert scaler is not None
            self.scaler = scaler
            self.hc_features = self.scaler.transform(features).astype('float32')
        self.labels = self.df['Label'].values

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_root, 'mask', row['category'], row['filename'])
        try: image = Image.open(img_path).convert('RGB')
        except: image = Image.new('RGB', (224, 224), (0, 0, 0))
        if self.transform: image = self.transform(image)
        hc_feat = torch.tensor(self.hc_features[idx])
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return image, hc_feat, label

# --- 5. ユーティリティ ---
def save_learning_curve(history, save_path):
    epochs = range(1, len(history['train_loss']) + 1)
    plt.figure(figsize=(14, 5))

    plt.subplot(1, 2, 1)
    plt.plot(epochs, history['train_loss'], 'r-', label='Train Loss')
    plt.plot(epochs, history['val_loss'], 'g-', label='Val Loss')
    plt.title('Training & Validation Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)

    plt.subplot(1, 2, 2)
    plt.plot(epochs, history['val_acc'], 'b-', label='Val Acc')
    plt.title('Validation Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()

# --- 6. メイン学習関数 ---
def train_model():
    BATCH_SIZE = 8
    EPOCHS = TARGET_EPOCH
    LR = 0.0001
    WD = 0.05
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    os.makedirs(SAVE_DIR, exist_ok=True)
    best_model_path = f'{SAVE_DIR}/best_fusion_model.pth'

# 訓練用 (データ拡張あり)
    train_transform = transforms.Compose([
        transforms.Resize((236, 236)),
        transforms.RandomRotation(degrees=15),
        transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
        transforms.RandomHorizontalFlip(p=0.5),
        
        # ★ NEW: カラージッターを追加
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
        
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    val_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    train_df, val_df = pd.read_csv(TRAIN_CSV_PATH), pd.read_csv(VAL_CSV_PATH)
    
    train_ds = CustomMultiModalDataset(train_df, IMG_ROOT, transform=train_transform, is_train=True)
    val_ds = CustomMultiModalDataset(val_df, IMG_ROOT, scaler=train_ds.scaler, transform=val_transform, is_train=False)
    
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
    
    model = FusionModel(num_classes=3).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    
    if USE_SCHEDULER:
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
        print("Scheduler: ENABLED (Cosine Annealing)")
    else:
        scheduler = None
        print("Scheduler: DISABLED (Fixed Learning Rate)")
    
    history = {'train_loss': [], 'val_loss': [], 'val_acc': []}
    best_acc = 0.0

    print(f"Starting Training on {DEVICE}...")

    for epoch in range(EPOCHS):
        # --- Train ---
        model.train()
        running_loss = 0.0
        for images, hc_feats, labels in train_loader:
            images, hc_feats, labels = images.to(DEVICE), hc_feats.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(images, hc_feats), labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * images.size(0)
        
        if scheduler is not None:
            scheduler.step()
            
        epoch_train_loss = running_loss / len(train_ds)

        # --- Val ---
        model.eval()
        v_running_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for images, hc_feats, labels in val_loader:
                images, hc_feats, labels = images.to(DEVICE), hc_feats.to(DEVICE), labels.to(DEVICE)
                outputs = model(images, hc_feats)
                v_loss = criterion(outputs, labels)
                v_running_loss += v_loss.item() * images.size(0)
                _, preds = torch.max(outputs, 1)
                total += labels.size(0)
                correct += torch.sum(preds == labels.data)
        
        epoch_val_loss = v_running_loss / len(val_ds)
        val_acc = (correct.double() / total).item()
        
        history['train_loss'].append(epoch_train_loss)
        history['val_loss'].append(epoch_val_loss)
        history['val_acc'].append(val_acc)
        
        curr_lr = optimizer.param_groups[0]['lr']
        print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f} | Val Acc: {val_acc:.4f} | LR: {curr_lr:.6f}")
        
        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), best_model_path)
            print(f"  --> Best Model Saved! (Acc: {best_acc:.4f})")
            
        if (epoch + 1) % 5 == 0: 
            save_learning_curve(history, f'{SAVE_DIR}/learning_curve.png')

    print(f"Training Complete. Best Val Acc: {best_acc:.4f}")
    save_learning_curve(history, f'{SAVE_DIR}/learning_curve.png')
    return history

if __name__ == "__main__":
    train_model()

Loading official ConvNeXt-Tiny pre-trained weights...
Scheduler: DISABLED (Fixed Learning Rate)
Starting Training on cuda...
Epoch 1/100 | Train Loss: 1.1303 | Val Loss: 1.0038 | Val Acc: 0.4375 | LR: 0.000100
  --> Best Model Saved! (Acc: 0.4375)
Epoch 2/100 | Train Loss: 0.9993 | Val Loss: 1.2813 | Val Acc: 0.3889 | LR: 0.000100
Epoch 3/100 | Train Loss: 0.9809 | Val Loss: 0.8684 | Val Acc: 0.5486 | LR: 0.000100
  --> Best Model Saved! (Acc: 0.5486)
Epoch 4/100 | Train Loss: 0.9439 | Val Loss: 0.8832 | Val Acc: 0.5486 | LR: 0.000100
Epoch 5/100 | Train Loss: 0.9823 | Val Loss: 0.8491 | Val Acc: 0.5903 | LR: 0.000100
  --> Best Model Saved! (Acc: 0.5903)
Epoch 6/100 | Train Loss: 0.9355 | Val Loss: 0.8250 | Val Acc: 0.6319 | LR: 0.000100
  --> Best Model Saved! (Acc: 0.6319)
Epoch 7/100 | Train Loss: 0.9291 | Val Loss: 0.8579 | Val Acc: 0.6181 | LR: 0.000100
Epoch 8/100 | Train Loss: 0.9136 | Val Loss: 0.8575 | Val Acc: 0.6042 | LR: 0.000100
Epoch 9/100 | Train Loss: 0.9042 | Val Loss